# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and perform preliminary analysis on the FAIR² dataset using the `mlcroissant` library. All references to record sets, fields, and columns are made via their `@id` fields according to the Croissant specification.

### Dataset Source
The Croissant schema describing this dataset is accessible here:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

To ensure consistent reference to dataset entities, we use their Croissant `@id` fields when extracting record sets or fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Explore the available record sets and fields in the dataset. All entities are referenced by their Croissant `@id`.

We will print the list of record sets and for each record set print its fields/columns and their respective `@id`.

In [ ]:
# List available record sets
print("Available record sets (by '@id'):")
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '')}")

print("\n---\n")
# Show fields for each record set
for rs in record_sets:
    print(f"Fields for record set '@id': {rs['@id']}")
    if 'fields' in rs and rs['fields']:
        for f in rs['fields']:
            print(f"    - {f['@id']}: {f.get('name', f['@id'])} (type: {f.get('dataType', 'unknown')})")
    elif 'columns' in rs and rs['columns']:
        for c in rs['columns']:
            print(f"    - {c['@id']}: {c.get('name', c['@id'])} (type: {c.get('dataType', 'unknown')})")
    print()
# For usage in future cells, collect all record set @ids
all_record_set_ids = [rs['@id'] for rs in record_sets]
pprint(all_record_set_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

Below, we will extract the data for all available record sets using their `@id`. You can select specific record sets, fields, or columns for targeted analysis. 

_Note: If the dataset contains only one main record set (typical for table-style datasets), focus on that one._

In [ ]:
# Extract all record set data into pandas DataFrames
dfs = {}
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))  # Each record is a dict (field @id : value)
    if records:
        dfs[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dfs[record_set_id])} records for record set: {record_set_id}")
        print(f"Columns (@id): {dfs[record_set_id].columns.tolist()}")
    else:
        print(f"No records found for record set: {record_set_id}")

# For convenience, choose the first available record set for further analysis
if len(dfs) > 0:
    primary_record_set_id = list(dfs.keys())[0]
    print(f"\nUsing primary record set: {primary_record_set_id}")
    display(dfs[primary_record_set_id].head())
else:
    print("No data tables found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping/categorizing data.

**Note:** All references to fields use the field or column `@id`.

Below, select and use the `@id` of a numeric column for filtering and normalization. Update parameters as appropriate for the dataset fields.

In [ ]:
# EDA: Choose a numeric field for demonstration.
# For mass spectrometry datasets, 'MW' (molecular weight) or 'abundance' are likely numeric fields.
# Edit these variables to match an actual numeric field @id from your dataset (see output above).

# Example (replace with real @id discovered in section 2 if different!):
numeric_field_id = None
group_field_id = None
for col in dfs[primary_record_set_id].columns:
    # Try to find an MW or abundance column by name/kind
    if 'mw' in col.lower() or 'MolecularWeight' in col or 'MW' in col:
        numeric_field_id = col
    if group_field_id is None and ('group' in col.lower() or 'class' in col.lower() or 'gene' in col.lower() or 'sample' in col.lower()):
        group_field_id = col
if numeric_field_id is None:
    numeric_field_id = dfs[primary_record_set_id].select_dtypes(include='number').columns[0]

print(f"Numeric field selected (by @id): {numeric_field_id}")

# Filtering: e.g., keep only proteins with MW above a threshold
threshold = dfs[primary_record_set_id][numeric_field_id].mean()  # Example: mean as threshold
filtered_df = dfs[primary_record_set_id][dfs[primary_record_set_id][numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized field for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally, group by a field if present
if group_field_id is not None and group_field_id in dfs[primary_record_set_id].columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between numeric fields, referring to fields via their `@id`.

For demonstration, we plot the distribution of the chosen numeric field and any grouped summaries.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(dfs[primary_record_set_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouped dataframe is available
    if 'grouped_df' in locals():
        plt.figure(figsize=(9, 4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated the process of loading a Croissant-based FAIR² dataset, exploring its structure, extracting tabular data by record set `@id`, and analyzing protein features such as molecular weight or abundance. All field references are made using their Croissant `@id` for reproducibility.

Further analysis can be performed by selecting other fields for EDA, integrating domain-specific analysis, or combining with external biological databases as needed.